In [1]:
import pathlib, sys
import matplotlib
matplotlib.use('Agg')
_root = pathlib.Path.cwd()
if not (_root / 'src').exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


# 04 · Arena: Classification

The full scoreboard: the four pipelines (`none`, `minimal`, `standard`, `aggressive`) across the classic TF-IDF model and the transformer, on two tasks — **sentiment** (where negation and casing carry the label) and **topic** (where content words dominate and cleaning is more benign).

In [2]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from src.cleaning.pipelines import PIPELINE_NAMES
from src.config import get_settings
from src.datasets.corpora import load_sentiment, load_topic
from src.evaluation.classification import evaluate_classification
from src.models.bow import BowClassifier
from src.models.transformer import TransformerEmbedder
from src.visualisation import plot_pipeline_scores, save_figure, save_json
seed = get_settings().seed
embedder = TransformerEmbedder()

def transformer_cv(texts, labels, pipeline, n_splits=5):
    emb = embedder.encode(texts, pipeline=pipeline)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    clf = LogisticRegression(max_iter=1000, random_state=seed)
    acc = cross_val_score(clf, emb, labels, cv=skf, scoring='accuracy').mean()
    f1 = cross_val_score(clf, emb, labels, cv=skf, scoring='f1_macro').mean()
    return float(acc), float(f1)

## Run every cell of the arena

4 pipelines × 2 models × 2 tasks = 16 measurements, each 5-fold cross-validated.

In [3]:
tasks = {'sentiment': load_sentiment(), 'topic': load_topic()}
rows = []
for task, (texts, labels) in tasks.items():
    for p in PIPELINE_NAMES:
        r = evaluate_classification(BowClassifier, texts, labels, pipeline=p)
        rows.append({'task': task, 'model': 'tfidf+logreg', 'pipeline': p,
                     'accuracy': r.accuracy, 'macro_f1': r.macro_f1})
        acc, f1 = transformer_cv(texts, labels, p)
        rows.append({'task': task, 'model': 'transformer', 'pipeline': p,
                     'accuracy': acc, 'macro_f1': f1})
arena = pd.DataFrame(rows)
save_json(arena, 'nb04_classification_arena.json')
arena.round(3)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,task,model,pipeline,accuracy,macro_f1
0,sentiment,tfidf+logreg,none,0.988,0.987
1,sentiment,transformer,none,1.000,1.000
2,sentiment,tfidf+logreg,minimal,0.988,0.987
3,sentiment,transformer,minimal,1.000,1.000
4,sentiment,tfidf+logreg,standard,0.975,0.975
5,sentiment,transformer,standard,0.988,0.987
6,sentiment,tfidf+logreg,aggressive,0.962,0.962
7,sentiment,transformer,aggressive,0.962,0.962
8,topic,tfidf+logreg,none,0.757,0.744
9,topic,transformer,none,1.000,1.000


## Sentiment scoreboard

In [4]:
sent = arena[arena['task'] == 'sentiment'].drop(columns='task')
fig = plot_pipeline_scores(sent, metric='macro_f1')
fig.axes[0].set_title('Sentiment: macro-F1 by pipeline and model')
save_figure(fig, 'nb04_arena_sentiment.png')
sent.sort_values('macro_f1', ascending=False).round(3)

,model,pipeline,accuracy,macro_f1
1,transformer,none,1.000,1.000
3,transformer,minimal,1.000,1.000
0,tfidf+logreg,none,0.988,0.987
2,tfidf+logreg,minimal,0.988,0.987
5,transformer,standard,0.988,0.987
4,tfidf+logreg,standard,0.975,0.975
6,tfidf+logreg,aggressive,0.962,0.962
7,transformer,aggressive,0.962,0.962


## Topic scoreboard

In [5]:
top = arena[arena['task'] == 'topic'].drop(columns='task')
fig = plot_pipeline_scores(top, metric='macro_f1')
fig.axes[0].set_title('Topic: macro-F1 by pipeline and model')
save_figure(fig, 'nb04_arena_topic.png')
top.sort_values('macro_f1', ascending=False).round(3)

,model,pipeline,accuracy,macro_f1
9,transformer,none,1.000,1.000
11,transformer,minimal,1.000,1.000
13,transformer,standard,1.000,1.000
15,transformer,aggressive,0.918,0.920
10,tfidf+logreg,minimal,0.757,0.744
8,tfidf+logreg,none,0.757,0.744
14,tfidf+logreg,aggressive,0.650,0.652
12,tfidf+logreg,standard,0.650,0.650


## Best pipeline per model and task

In [6]:
best = (arena.sort_values('macro_f1', ascending=False)
        .groupby(['task', 'model'], as_index=False).first())
save_json(best, 'nb04_best_pipeline.json')
best.round(3)

,task,model,pipeline,accuracy,macro_f1
0,sentiment,tfidf+logreg,none,0.988,0.987
1,sentiment,transformer,none,1.000,1.000
2,topic,tfidf+logreg,minimal,0.757,0.744
3,topic,transformer,minimal,1.000,1.000


## Takeaway

The verdict is model- and task-dependent, which is the whole point. Cleaning does the least damage on topic (content words dominate and survive cleaning) and the most on sentiment (negation and casing are the label). The aggressive pipeline — the kitchen sink every tutorial reaches for — is never the best choice for the transformer. There is no single cleaning pipeline that is right; you have to measure it on your model and your task.